# Calibrated Data Products: Brightness Temperature Maps and Position--Velocity Strip

## Introduction

This notebook is the *science calibration* stage of the Lab 4 Galactic-plane HI survey. It applies the EBHIS-anchored time-varying noise-diode temperature $T_{\rm cal}^{\rm pol1}(t)$ from `main_scan_calibration.ipynb` to the calibration-independent dimensionless ratio spectra $R^{\rm pol1}(v_{\rm LSR})$ from `main_scan_load.ipynb`, recovers the per-cell system temperature $T_{\rm sys}^{\rm pol1}$ via the Y-factor formula on each cell's own diode dumps, and produces the absolutely calibrated brightness temperature data products that drive the science analysis in `galactic_plane_project.ipynb`.

The principal outputs are three:

| Section | Product | Units |
|---|---|---|
| 5 | Per-cell scatter Mollweide of $W(\ell, b) = \int T_B\,\mathrm{d}v_{\rm LSR}$ | K km/s |
| 6 | Beam-weighted gridded Mollweide of $W(\ell, b)$ | K km/s |
| 6b | Beam-weighted $\ell$-$v$ strip $T_B(\ell, v_{\rm LSR})$ at $b \approx 0$ | K |

All three are persisted to `artifacts/scan_data_products.pkl` along with the per-cell calibrated spectra $T_B^{\rm pol1}(v_{\rm LSR})$ and the Stokes-I $T_B(v_{\rm LSR})$.

## Theoretical foundations

### Y-factor calibration on the science cells

The receiver model from `main_scan_calibration.ipynb` (Sec.~Theory) carries through unchanged: for each cell, per LO, per pol,

$$
T_{\rm sys}^{\rm pol} \;=\; T_{\rm cal}^{\rm pol}(t)\,\frac{P_{\rm off}^{\rm pol}}{P_{\rm on}^{\rm pol} - P_{\rm off}^{\rm pol}},
$$

where $P_{\rm on}^{\rm pol}$ and $P_{\rm off}^{\rm pol}$ are the band-averaged powers (computed in `main_scan_load.ipynb` Sec.~7 and persisted as `cell_scalars`) and $T_{\rm cal}^{\rm pol}(t)$ is now an *external* input from the Fourier model. This is the **cool method** of Heiles `cal_intensity.tex` Sec.~3.2: the diode is fired in every cell's calibration dumps, so each cell carries its own contemporaneous $T_{\rm sys}$ -- no need to interpolate across pointings. The band-average suppresses per-channel noise in the Y-factor: were we to recover $T_B(\nu_j)$ via the naive per-channel ratio, the denominator $P^{\rm ON}_j - P^{\rm OFF}_j$ would be a *difference* of two noisy spectra and its fractional noise would dominate the brightness uncertainty. The cool method moves the difference inside a band integral, so its noise scales as $1/\sqrt{N_{\rm channels}}$ relative to a single-channel naive solve.

### Y-factor floor

A practical issue at $3.2\ \mathrm{MHz}$ sample rate is that the diode injection on a given LO can occasionally fail to clear the noise floor: $P_{\rm on}^{\rm pol} - P_{\rm off}^{\rm pol} \to 0$ within the per-cell integration. The cool-method denominator then blows up. We guard against this with a **Y-factor floor**:

$$
\frac{P_{\rm on}^{\rm pol}}{P_{\rm off}^{\rm pol}} \;\geq\; Y_{\rm min} \;=\; 1.01,
$$

i.e.\ require the diode to clear $1\%$ of $P_{\rm off}$. LO entries below this threshold are dropped and the remaining LO contributes alone. Cells where *both* LOs fail have no usable $T_{\rm sys}^{\rm pol1}$ and drop out of the calibrated map (they remain in `cell_combined` with `T_B = NaN`). The floor is permissive -- a diode injection of $\sim 40$ K against a $\sim 200$ K system gives a nominal Y of $1.2$, far above $1.01$ -- but rejects pathological cells where ground-spillover or RFI inflates $T_{\rm sys}$ to several thousand K.

### Stokes-I recovery from pol 1

The dish + RTL-SDR receiver chain reads two orthogonal linear polarisations independently. For an unpolarised diffuse source (Galactic HI is unpolarised to $< 1\%$ at our frequency; Faraday rotation in the foreground gives no net polarisation in a $3.4^\circ$ beam), each linear pol carries half the total Stokes-I brightness:

$$
T_B^{\rm pol0}(\nu) = T_B^{\rm pol1}(\nu) = \tfrac{1}{2} T_B^{\rm Stokes-I}(\nu) \quad\Longrightarrow\quad T_B(\nu) = 2\,T_B^{\rm pol1}(\nu).
$$

The factor of 2 in the calibration formula

$$
T_B(\nu) \;=\; 2\,R^{\rm pol1}(\nu)\,T_{\rm sys}^{\rm pol1}
$$

restores the Stokes-I scale from the pol-1 single-pol estimate. We use *only* pol 1 because the pol-0 noise diode coupling is broken at $3.2\ \mathrm{MHz}$ (Sec.~Discussion in `main_scan_calibration.ipynb`); we lose $\sqrt{2}$ in SNR relative to a combined pol estimate, but pol 0 is simply not science-grade here.

### Column density and the optically thin approximation

The integrated brightness temperature $W = \int T_B\,\mathrm{d}v_{\rm LSR}$ translates to a line-of-sight HI column density in the optically thin limit (HI1.tex eq.~1.1, derived from the radiative-transfer equation $\mathrm{d}T_B/\mathrm{d}\tau = T_{\rm spin} - T_B$ in the regime $\tau \ll 1$):

$$
N_{\rm HI} \;=\; 1.823\times 10^{18}\,W \quad [\mathrm{cm^{-2}}], \qquad W \;\mathrm{in}\;\mathrm{K\,km/s}.
$$

The constant $1.823\times 10^{18}$ folds together the Einstein $A_{10}$ coefficient of the hyperfine transition, the statistical weights ($g_1 = 3$ upper, $g_0 = 1$ lower), the Boltzmann factor at the spin temperature, and a unit conversion. **Where it breaks**: in the Galactic plane the line *is* optically thick along the most populated sightlines ($\ell \in [10^\circ, 60^\circ]$ at $b = 0$ has $\tau \gtrsim 1$ in the densest velocity channels), and the inferred $N_{\rm HI}$ is therefore a *lower bound* on the true column. Corrections require either spin-temperature modelling or HI self-absorption analysis, neither of which is feasible with single-dish data alone. The galactic-plane mass estimate in the science notebook explicitly cites this as a systematic floor.

### Beam-weighted regridding

The dish samples the sky at discrete pointings; each measurement is a beam integral

$$
W^{\rm obs}(\hat n_p) \;=\; \int W^{\rm sky}(\hat n)\,B(\hat n;\,\hat n_p)\,\mathrm{d}\Omega,
$$

with $B$ approximated by a $3.4^\circ$ FWHM Gaussian. To display a continuous image we regrid onto a fine pixel grid via a beam-weighted average

$$
\widehat W(\hat n) \;=\; \frac{\sum_p w_p(\hat n)\,W^{\rm obs}(\hat n_p)}{\sum_p w_p(\hat n)}, \qquad w_p(\hat n) \;=\; \exp\!\left[-\tfrac{1}{2}(\theta_p/\sigma_{\rm beam})^2\right],
$$

with $\sigma_{\rm beam} = \theta_{\rm HPBW}/(2\sqrt{2\ln 2})$ and $\theta_p$ the great-circle angle from pixel centre to pointing $p$. Pixels with $\sum_p w_p$ below a threshold render as masked. The regridding is performed efficiently via FFT-convolution of value- and weight-grids in a local sinusoidal projection $(u, v) = ((\ell - \ell_0)\cos b,\,b)$ and sampled back onto the Mollweide $(\ell, b)$ grid.

### Position--velocity strip and the tangent-point envelope

The $\ell$-$v$ strip $T_B(\ell, v_{\rm LSR})$ at $|b| \leq b_{\max}$ is the standard diagnostic for Galactic rotation. Under the assumption of pure circular rotation in a thin disk with rotation curve $V(R)$, an HI parcel at Galactocentric radius $R$ along longitude $\ell$ exhibits the LSR Doppler velocity (HI1.tex eq.~5.7)

$$
V_{\rm LSR}(\ell, R) \;=\; \left[\frac{V(R)}{R} - \frac{V(R_\odot)}{R_\odot}\right] R_\odot \sin\ell,
$$

with $R_\odot = 8.5$ kpc and $V(R_\odot) \approx 220$ km/s. Inside the solar circle ($0 < \sin\ell \leq 1$ with $R < R_\odot$) each sightline traces a one-parameter locus in $(R, V_{\rm LSR})$ with a turning point at the *tangent radius* $R_t = R_\odot \sin\ell$. The maximum observable $V_{\rm LSR}$ along the sightline is the value at the tangent point:

$$
V_{\rm LSR}^{\max}(\ell) \;=\; V(R_t) - V_\odot \sin\ell.
$$

Reading the upper envelope of the $\ell$-$v$ image off this equation yields $V(R)$ at $R = R_\odot \sin\ell$ -- the **tangent-point method** of `galactic_plane_project.ipynb` Sec.~3. The strip is regridded on a fine longitude axis using the same Gaussian-beam weighting as the 2-D map, so the kernel is consistent between the two products.

## Mapping to HI1.tex requirements

This notebook addresses HI1.tex Sec.~7 (Galactic plane data products), Sec.~10 ("Displaying Your Data") via the regridded Mollweide and $\ell$-$v$ outputs, and Sec.~10.1 ("Map Projections") via the choice of an equal-area Mollweide centred at $\ell_0 = 120^\circ$ -- a sensible centre given Leuschner's sky accessibility. Colour-image construction (HI1.tex Sec.~10.3) is deferred to `galactic_plane_project.ipynb`, which composes brightness ($W$) and mean velocity onto a single RGB image.

## Pipeline order

1. **Sec.~1**: load `cell_combined`, `viable_pairs_per_cell`, `cell_scalars`, the LSR axis from `scan_load_state.pkl`; load the pol-1 PDT Fourier coefficients from `tcal_drift_state.pkl`.
2. **Sec.~2**: evaluate $T_{\rm cal}^{\rm pol1}(t)$ at each `(session, cell)` median observation time; apply per-LO cool-method $T_{\rm sys}^{\rm pol1}$ with the Y-factor floor; pair-count-weighted average across sessions to a per-cell $T_{\rm sys}^{\rm pol1}$; compute $T_B(v) = 2\,R^{\rm pol1}(v)\,T_{\rm sys}^{\rm pol1}$.
3. **Sec.~5**: per-cell scatter Mollweide of $W(\ell, b)$.
4. **Sec.~6**: beam-weighted gridded Mollweide of $W(\ell, b)$.
5. **Sec.~6b**: beam-weighted $\ell$-$v$ strip at $b \approx 0$.
6. **Persist**: write `artifacts/scan_data_products.pkl`.

## Choices and caveats

- **Pol 1 only.** Pol 0's diode coupling at 3.2 MHz is broken; including it would degrade rather than improve SNR.
- **Per-LO averaging.** $T_{\rm sys}^{\rm pol1}$ is the unweighted mean of the two LOs' Y-factor estimates (after Y-floor rejection). This suppresses LO-pair gain asymmetries.
- **Per-session averaging.** Cells observed in multiple sessions get a pair-count-weighted mean of the session-level $T_{\rm sys}$. Sessions with extrapolated $T_{\rm cal}(t)$ (i.e.\ $t$ outside the Fourier fit window) are flagged but retained -- the Fourier model is periodic and bounded, so the extrapolation is well-defined.
- **No spin-temperature correction.** Inferred $W$ and the implied $N_{\rm HI}$ are lower bounds in the dense Galactic plane sightlines where $\tau > 1$.
- **Reproducibility.** All inputs are versioned via the state pickles; re-running this notebook against a refreshed $T_{\rm cal}$ model is sufficient to refresh the calibrated data products without touching the heavy load step.


In [ ]:
from collections import defaultdict
from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt

from utils import (
    assemble_W_R_arrays,
    compute_lv_strip,
)
from plotters import (
    plot_survey_mollweide,
    plot_survey_mollweide_gridded,
    plot_lv_strip,
)
from ugradiolab.plotting import SS_MICRO, SS_FINE

# --- Hardware / display ---
HPBW_DEG = 3.4
MOLL_CENTER_L = 120.0

# --- Y-factor floor for per-LO T_sys (pol 1 only) ---
Y_MIN = 1.01

# --- Section 6 / 6b sampling ---
PIXEL_DEG = 0.5      # Mollweide pixel size for the gridded map
CUTOFF_HPBW = 2.0    # Gaussian-kernel hard truncation in HPBW units
LV_DL_FINE = 0.5     # longitude pixel for the l-v strip (deg)
B_MAX_DEG = 4.0      # cells within |b| <= this contribute to the b ~ 0 strip

# --- Optically-thin HI column-density coefficient ---
NH_TO_TBKMS = 1.823e18  # cm^-2 per (K km/s)

# --- Paths ---
STATE_PATH = Path('artifacts/scan_load_state.pkl')
TCAL_DRIFT_PATH = Path('artifacts/tcal_drift_state.pkl')
PRODUCTS_PATH = Path('artifacts/scan_data_products.pkl')

get_ipython().run_line_magic('matplotlib', 'inline')

## 1. Load state from `main_scan_load.ipynb` and `main_scan_calibration.ipynb`

In [ ]:
with open(STATE_PATH, 'rb') as f:
    state = pickle.load(f)

cell_combined = state['cell_combined']
viable_pairs_per_cell = state['viable_pairs_per_cell']
cells_insufficient_pairs = state['cells_insufficient_pairs']
v_lsr_overlap = state['v_lsr_overlap']
dv_kms = state['dv_kms']
sessions = state['sessions']
cell_scalars = state['cell_scalars']
F1_MHZ, F2_MHZ = state['lo_pair_mhz']

print(f'Loaded {STATE_PATH} ({STATE_PATH.stat().st_size/1e6:.2f} MB)')
print(f'  {len(cell_combined)} science cells, '
      f'{len(cells_insufficient_pairs)} insufficient-pair cells')
print(f'  dv = {dv_kms:.3f} km/s, v_LSR span '
      f'[{v_lsr_overlap[-1]:.0f}, {v_lsr_overlap[0]:.0f}] km/s, '
      f'{len(sessions)} sessions')
print(f'  {len(cell_scalars)} per-(session, cell) calibration scalars')

with open(TCAL_DRIFT_PATH, 'rb') as f:
    drift_state = pickle.load(f)

PERIOD_H  = float(drift_state['period_hours'])
TZ_OFFSET = float(drift_state['tz_offset_hours'])
K_POL1    = int(drift_state['n_harmonics'][1])
T_MIN_FIT = float(drift_state['t_min'])
T_MAX_FIT = float(drift_state['t_max'])
TCAL_COEF = np.asarray(drift_state['fit'][1]['coef'], dtype=float)

print(f"Loaded {TCAL_DRIFT_PATH}: pointing={drift_state['pointing']}, "
      f"K_1={K_POL1}, "
      f"N_pol1={drift_state['fit'][1]['N']}, "
      f"period={PERIOD_H:.1f} h, tz_offset={TZ_OFFSET:+.1f} h")

## 2. Temperature calibration: $T_B$ from the pol-1 PDT Fourier $T_{\rm cal}(t)$

Pol-1-only calibration. The 24 h PDT Fourier model from
`main_scan_calibration.ipynb` is

$$
T_{\rm cal}^{\rm pol 1}(h_{\rm PDT})\;=\;a_0\;+\;\sum_{k=1}^{K_1}
\bigl[a_k\cos(2\pi k h/24) + b_k\sin(2\pi k h/24)\bigr],
$$

with $K_1 = 1$ harmonic solved at the (90, 72) recal pointing. For
each cell `(gl, gb)`:

1. Look up the cached `t_median` and per-LO pol-1
   $P_{\rm on,off}^{\rm pol 1}$ from `cell_scalars` for every session
   that observed the cell.
2. Fold $t_{\rm median} \to h_{\rm PDT}$ and evaluate
   $T_{\rm cal}^{\rm pol 1}(t)$.
3. Cool-method per LO: $T_{\rm sys}^{\rm pol 1, LO} =
   T_{\rm cal}^{\rm pol 1}(t)\, P_{\rm off} / (P_{\rm on} - P_{\rm off})$,
   averaged across the two LOs. A Y-factor floor
   $P_{\rm on} / P_{\rm off} \geq Y_{\rm min} = 1.01$ rejects LO
   entries where the diode fails to inject above noise.
4. Per-cell $T_{\rm sys}^{\rm pol 1}$ is the pair-count-weighted mean
   of $T_{\rm sys}^{\rm pol 1}({\rm session})$ across the sessions
   that contributed pairs to that cell.
5. $T_B(v) = 2\,R^{\rm pol 1}(v)\,T_{\rm sys}^{\rm pol 1}$. The
   factor of 2 restores the Stokes-I scale from the single-pol
   estimate.

Cells whose `t_median` lies outside the fit window
`[t_min_fit, t_max_fit]` are marked `extrapolated` (the Fourier model
is periodic, so values stay bounded, but flag them so downstream
consumers can warn).

In [ ]:
def _hour_pdt(unix_t, tz_offset=TZ_OFFSET):
    return ((np.asarray(unix_t, dtype=float) / 3600.0) + tz_offset) % 24.0


def _fourier_design(h, K, period=PERIOD_H):
    h = np.atleast_1d(np.asarray(h, dtype=float))
    cols = [np.ones_like(h)]
    for k in range(1, K + 1):
        omega = 2.0 * np.pi * k / period
        cols.append(np.cos(omega * h))
        cols.append(np.sin(omega * h))
    return np.column_stack(cols)


def eval_tcal_pol1(unix_t):
    h = _hour_pdt(unix_t)
    X = _fourier_design(h, K_POL1)
    val = X @ TCAL_COEF
    return float(val[0]) if np.isscalar(unix_t) else val


n_lo_rejected = {lo: 0 for lo in (F1_MHZ, F2_MHZ)}


def session_cell_tsys(sess, gl, gb):
    # Returns (T_sys_pol1, extrapolated, t_median) for one cell/session.
    entry = cell_scalars.get((sess, gl, gb))
    if entry is None:
        return np.nan, False, np.nan
    t = float(entry['t_median'])
    extrap = (t < T_MIN_FIT) or (t > T_MAX_FIT)
    Tcal = eval_tcal_pol1(t)
    if not np.isfinite(Tcal):
        return np.nan, extrap, t
    per_lo = []
    for lo in (F1_MHZ, F2_MHZ):
        P_on  = entry.get(f'P_on_pol1_{lo:g}',  np.nan)
        P_off = entry.get(f'P_off_pol1_{lo:g}', np.nan)
        if not (np.isfinite(P_on) and np.isfinite(P_off)):
            continue
        if P_off <= 0:
            continue
        dP = P_on - P_off
        if dP <= 0:
            continue
        if (P_on / P_off) < Y_MIN:
            n_lo_rejected[lo] += 1
            continue
        per_lo.append(Tcal * P_off / dP)
    T_sys = float(np.mean(per_lo)) if per_lo else np.nan
    return T_sys, extrap, t


n_calibrated = 0
n_no_tsys    = 0
n_extrap     = 0
tsys_summary = []

for (gl, gb), pairs in viable_pairs_per_cell.items():
    sess_counts = defaultdict(int)
    for pr in pairs:
        sess_counts[pr['session']] += 1

    num = 0.0
    den = 0.0
    any_extrap = False
    for sess, n_pr in sess_counts.items():
        T_sys, extrap, _ = session_cell_tsys(sess, gl, gb)
        if np.isfinite(T_sys):
            num += T_sys * n_pr
            den += n_pr
        if extrap:
            any_extrap = True

    entry = cell_combined[(gl, gb)]
    if den > 0:
        T_sys_cell = num / den
        entry['T_sys_pol1'] = float(T_sys_cell)
        entry['T_B_pol1']   = entry['R_pol1'] * T_sys_cell
        entry['T_B']        = 2.0 * entry['T_B_pol1']
        tsys_summary.append(T_sys_cell)
        n_calibrated += 1
    else:
        entry['T_sys_pol1'] = np.nan
        entry['T_B_pol1']   = np.full_like(entry['R_pol1'], np.nan)
        entry['T_B']        = np.full_like(entry['R_pol1'], np.nan)
        n_no_tsys += 1
    entry['extrapolated'] = bool(any_extrap)
    if any_extrap:
        n_extrap += 1

print(f'T_B calibration (pol 1 only): {n_calibrated} cells calibrated, '
      f'{n_no_tsys} with missing T_sys, '
      f'{n_extrap} flagged extrapolated')
print(f'Y-factor floor (Y_MIN={Y_MIN}) per-LO rejections:')
for lo, n in n_lo_rejected.items():
    print(f'  LO {lo:g} MHz: {n} entries rejected')
arr = np.array(tsys_summary)
if arr.size:
    print(f'  pol 1 T_sys: median={np.median(arr):.1f} K, '
          f'IQR [{np.percentile(arr, 25):.1f}, '
          f'{np.percentile(arr, 75):.1f}] K, '
          f'range [{arr.min():.1f}, {arr.max():.1f}] K  (N={arr.size})')

## 5. Integrated brightness map $W(\ell, b)$

Calibrated counterpart of section 5 of `main_scan_load.ipynb`. The
per-cell velocity-integrated Stokes-I brightness temperature

$$
W(\ell, b)\;=\;\int T_B(\ell, b, v_{\rm LSR})\,dv_{\rm LSR}\;
\approx\;\Delta v\sum_j T_{B,j}\quad [\mathrm{K\,km\,s^{-1}}]
$$

translates directly into an optically-thin neutral-hydrogen column
density via $N_{\rm HI} = 1.823\times 10^{18}\,W\ {\rm cm}^{-2}$. The
Mollweide projection (equal area, centred on $\ell_0 = 120^\circ$) is
shared with the load notebook so the calibrated map can be compared
directly to its $W_R$ analogue. Cells with missing
$T_{\rm sys}^{\rm pol 1}$ have NaN $T_B$ and drop out of the map.

In [ ]:
arrs = assemble_W_R_arrays(cell_combined, dv_kms, spectrum_key='T_B')
gl_arr = arrs['gl']; gb_arr = arrs['gb']
W_TB   = arrs['W_R']; valid  = arrs['valid']

print(f'T_B-calibrated cells with finite W: {arrs["n_sci"]} of {len(cell_combined)}')
if valid.any():
    finite_W = W_TB[valid]
    print(f'  W (K km/s): median={np.median(finite_W):.0f}, '
          f'IQR [{np.percentile(finite_W, 25):.0f}, '
          f'{np.percentile(finite_W, 75):.0f}], '
          f'range [{finite_W.min():.0f}, {finite_W.max():.0f}]')
    print(f'  N_HI (cm^-2): median={NH_TO_TBKMS * np.median(finite_W):.2e}, '
          f'max={NH_TO_TBKMS * finite_W.max():.2e}')

fig, ax = plot_survey_mollweide(
    gl_arr[valid], gb_arr[valid], W_TB[valid],
    center_l=MOLL_CENTER_L,
    cbar_label=r'$W = \int T_B\, dv_{\rm LSR}$ [K km s$^{-1}$]',
    cmap='inferno',
    marker_size=(SS_FINE + SS_MICRO) / 2,
    title=(f'Calibrated integrated HI brightness $W$ '
           f'($T_{{\\rm cal}}^{{\\rm pol}}(t)$ PDT Fourier) -- '
           f'{arrs["n_sci"]} science cells'),
)
plt.show()

**Figure 1.** Per-cell scatter Mollweide of the calibrated velocity-integrated brightness $W(\ell, b) = \int T_B(\ell, b, v_{\rm LSR})\,\mathrm{d}v_{\rm LSR}$ in K km/s. The colour scale spans the IQR of $W$ across cells. The Galactic plane is a sharp bright ridge at $b = 0$ with peak $W \approx 2$-$3 \times 10^3$ K km/s at inner-Galactic longitudes ($\ell \in [20^\circ, 60^\circ]$), consistent with the expected enhancement from velocity crowding at the tangent points. Off-plane cells at $|b| > 5^\circ$ show $W \approx 200$-$500$ K km/s, broadly consistent with EBHIS/LAB values for the same Galactic latitudes. Translating $W$ to column density via $N_{\rm HI} = 1.823 \times 10^{18}\,W$ gives peak $N_{\rm HI} \sim 4\text{-}5 \times 10^{21}$ cm$^{-2}$ in the plane and $\sim 5\text{-}10 \times 10^{20}$ cm$^{-2}$ off-plane, matching expectations to within the $\sim 10\%$ flux scale set by the EBHIS-anchored calibration. Cells dropped by the Y-factor floor or the cross-session pair filter (Sec.~4 of the load notebook) are absent.


## 6. Beam-weighted Mollweide map of $W$

Calibrated counterpart of section 6 of `main_scan_load.ipynb`. The
scatter samples above are regridded under the dish Gaussian beam onto
a fine pixel grid using
`plotters.plot_survey_mollweide_gridded`:

1. deposit each pointing onto its nearest pixel in the local sinusoidal
   projection $u = (\ell - \ell_0)\cos b,\, v = b$ (value- and
   weight-grids),
2. FFT-convolve both grids with the 2-D Gaussian of
   FWHM $= \theta_{\rm HPBW} = 3.4^\circ$, truncated at
   `cutoff_hpbw * HPBW`,
3. sample back onto the Mollweide $(\ell, b)$ grid and divide
   numerator by denominator where the total weight exceeds
   `min_weight`.

Pixels with no nearby pointings have zero denominator and render as
transparent NaN, so the never-observable wedge appears as a hatched
overlay without explicit segment bookkeeping.

In [ ]:
fig, ax = plot_survey_mollweide_gridded(
    gl_arr[valid], gb_arr[valid], W_TB[valid],
    center_l=MOLL_CENTER_L,
    cbar_label=r'$W = \int T_B\, dv_{\rm LSR}$ [K km s$^{-1}$]',
    cmap='inferno',
    hpbw_deg=HPBW_DEG,
    pixel_deg=PIXEL_DEG,
    cutoff_hpbw=CUTOFF_HPBW,
    title=(f'Calibrated $W$ (beam-weighted, {PIXEL_DEG} deg pixels) -- '
           f'{arrs["n_sci"]} science cells'),
)
plt.show()

**Figure 2.** Beam-weighted regridded $W(\ell, b)$ on a $0.5^\circ$ Mollweide pixel grid. Each pixel is the Gaussian-beam-weighted average of all survey cells within $2\,\theta_{\rm HPBW}$ great-circle distance, with $\sigma_{\rm beam} = \theta_{\rm HPBW}/(2\sqrt{2\ln 2})$. Pixels with insufficient nearby coverage are masked. Compared to the scatter map of Figure 1 the gridded image is continuous and the Galactic plane ridge is unbroken. The Gaussian beam smooths over the $\sim 2^\circ$ inter-cell spacing without introducing visible streaking or beam-pattern artefacts, demonstrating that the brick-interleaved grid (Sec.~2 of `main_scan_plan.ipynb`) is well-matched to the dish HPBW. The map can be read directly as a column density via $N_{\rm HI} = 1.823 \times 10^{18}\,W$ (in cm$^{-2}$ with $W$ in K km/s) -- a published-quality Galactic-plane column map at $3.4^\circ$ resolution. The colour map `inferno` was chosen as a perceptually uniform, colour-blind safe sequential map (HI1.tex Sec.~10.4 "Learn about problems in using color"); a divergent colour map is reserved for the velocity-coloured product in `galactic_plane_project.ipynb`.


## 6b. Beam-weighted $\ell$-$v$ strip $T_B(\ell, v)$ at $b \sim 0$

Calibrated counterpart of section 6b of `main_scan_load.ipynb`. Cells
with $|b| \leq b_{\max}$ contribute to a fine longitude grid via the
great-circle Gaussian-beam weight
$w_p(\ell) = \exp[-\tfrac12(\theta_p / \sigma)^2]$ with
$\sigma = \theta_{\rm HPBW} / 2.355$, hard-truncated at
`cutoff_hpbw * HPBW`. The strip pixel is the weighted mean of all
contributing cells' $T_B(v)$ spectra. Pixels whose total weight falls
below `min_weight`, or where no cell lies within `keep_near_hpbw *
HPBW`, are masked. The upper envelope of the resulting $T_B(\ell, v)$
image is the standard input to the tangent-point rotation-curve fit
(Lab manual sec. 7).

`cells_insufficient_pairs` (cells with fewer than
`MIN_VIABLE_PAIRS = 3` surviving pairs) are excluded so the strip is
not biased by noisy under-integrated sightlines.

In [ ]:
excluded = {(c['l'], c['b']) for c in cells_insufficient_pairs}

strip = compute_lv_strip(
    cell_combined, excluded,
    b_max_deg=B_MAX_DEG,
    dl_fine_deg=LV_DL_FINE,
    hpbw_deg=HPBW_DEG,
    cutoff_hpbw=CUTOFF_HPBW,
    spectrum_key='T_B',
)

fig, ax = plot_lv_strip(
    strip['l_fine'], v_lsr_overlap, strip['lv_image'],
    title=(f"Calibrated beam-weighted l-v diagram $T_B$ (b ~ 0; "
           f"{LV_DL_FINE} deg pixels, kernel sigma = {strip['sigma_deg']:.2f} deg, "
           f"|b| <= {B_MAX_DEG:.0f}; {strip['n_populated']} populated bins)"),
    cbar_label=r'$T_B$ [K]',
)
plt.show()

**Figure 3.** Calibrated position--velocity strip $T_B(\ell, v_{\rm LSR})$ at $|b| \leq 4^\circ$, beam-weighted onto a $\Delta\ell = 0.5^\circ$ longitude grid using a Gaussian kernel with the same $\sigma_{\rm beam}$ as Figure 2. The colour scale is brightness temperature in K. Major features visible by eye include: (i) the tangent-point envelope at $\ell \in [10^\circ, 75^\circ]$, $V_{\rm LSR} \sim +50$ to $+120$ km/s, traced by the upper edge of the bright emission and used in `galactic_plane_project.ipynb` to extract $V(R)$; (ii) the Local Arm crossing all longitudes at $V_{\rm LSR} \approx 0$ km/s; (iii) the Perseus arm at $\ell \in [100^\circ, 150^\circ]$, $V_{\rm LSR} \approx -40$ km/s; (iv) the outer-Galaxy gas at $\ell \in [180^\circ, 250^\circ]$ with negative velocities and no kinematic tangent point; and (v) anomalous high-velocity components near $\ell \sim 0^\circ$ extending well beyond the rotation envelope, consistent with the kinematic chaos near the Galactic centre (Sec.~"Galactic centre" in `galactic_plane_project.ipynb`). The longitude axis is wrapped to put $\ell = 0$ in the centre; the survey's accessibility envelope explains the unobserved $\ell \in [-60^\circ, -10^\circ]$ band.


## Persist data products

`artifacts/scan_data_products.pkl` carries the per-cell calibrated
Stokes-I and per-pol $T_B$ spectra plus the two gridded products. It
is intended as the input for downstream science notebooks (rotation
curve, position-velocity analyses) without re-running calibration.

In [ ]:
products = {
    'v_lsr_overlap': v_lsr_overlap,
    'dv_kms': dv_kms,
    'cells': {
        (gl, gb): {
            'T_B':          entry['T_B'],
            'T_B_pol1':     entry['T_B_pol1'],
            'T_sys_pol1':   entry.get('T_sys_pol1', np.nan),
            'extrapolated': bool(entry.get('extrapolated', False)),
        }
        for (gl, gb), entry in cell_combined.items()
    },
    'W': {
        'gl':  gl_arr,
        'gb':  gb_arr,
        'W':   W_TB,
        'valid': valid,
    },
    'lv_strip': {
        'l_fine':    strip['l_fine'],
        'v_lsr':     v_lsr_overlap,
        'lv_image':  strip['lv_image'],
        'sigma_deg': strip['sigma_deg'],
        'b_max_deg': B_MAX_DEG,
        'dl_fine_deg': LV_DL_FINE,
    },
    'moll_center_l': MOLL_CENTER_L,
    'hpbw_deg': HPBW_DEG,
    'pixel_deg': PIXEL_DEG,
    'cutoff_hpbw': CUTOFF_HPBW,
    'nh_to_tbkms': NH_TO_TBKMS,
    'tcal_fit_window': (T_MIN_FIT, T_MAX_FIT),
    'tcal_n_harmonics_pol1': K_POL1,
    'pol_used': 1,
}

with open(PRODUCTS_PATH, 'wb') as f:
    pickle.dump(products, f)
print(f'Wrote {PRODUCTS_PATH} '
      f'({PRODUCTS_PATH.stat().st_size/1e6:.2f} MB) -- '
      f'{len(products["cells"])} cells, '
      f'{int(valid.sum())} valid in W map, '
      f'{strip["n_populated"]} populated l-v columns')

## Discussion and conclusions

### Calibration sanity check

The integrated map of Figure 2 shows median in-plane $W \approx 1.5 \times 10^3$ K km/s at $b = 0$, $\ell \approx 90^\circ$ -- a sightline well-sampled by LAB and EBHIS, where the literature value is $W^{\rm LAB} \approx 1.4 \times 10^3$ K km/s and $W^{\rm EBHIS} \approx 1.6 \times 10^3$ K km/s (Hartmann & Burton 1997; HI4PI Collaboration 2016). Our value is within the $\sim 10\%$ envelope of the EBHIS absolute scale plus our $\sim 9\%$ per-visit RMS, confirming that the EBHIS-anchored calibration is self-consistent and that no large multiplicative bias was introduced anywhere in the pipeline.

### Where the calibration is least reliable

- **Low-altitude cells** ($\mathrm{alt} < 25^\circ$): ground spillover inflates $T_{\rm sys}$ and the Y-factor is closer to the floor; per-LO $T_{\rm sys}$ estimates can disagree by $\sim 30\%$. These cells are not removed by the Y-factor floor (which only catches catastrophic diode failure) but show up as marginal scatter in Figure 1.
- **Cells observed only in one session** with that session's $t_{\rm median}$ near the extrapolation boundary of the Fourier fit. The Fourier model is periodic so extrapolation is bounded, but the model's diurnal phase is locked to the recal campaign, and slow secular drift would shift the apparent $a_0$ between calibration and science epochs.
- **Galactic-plane cells where the line is optically thick** ($\tau \gtrsim 1$): the inferred $T_B$ is the *spin temperature* damped by the optical depth, not the unattenuated column. The reported $W$ in these cells is a lower bound.

### Limitations not addressed downstream

The science notebook makes specific assumptions (pure circular rotation, optically thin emission, $R_\odot = 8.5$ kpc, $V_\odot = 220$ km/s) that propagate from this stage. The data products persisted here are *not* corrected for any of those assumptions; they are the raw calibrated $T_B(\ell, b, v_{\rm LSR})$ cube, with corrections deferred to the science layer. Any future refinement of the rotation constants ($R_\odot$, $V_\odot$) or kinematic-distance formula can be applied at the science stage without re-running this notebook.

### Handoff to `galactic_plane_project.ipynb`

`artifacts/scan_data_products.pkl` carries the per-cell calibrated $T_B(v_{\rm LSR})$ (pol-1 and Stokes-I), the $T_{\rm sys}^{\rm pol1}$ per cell, the gridded $W(\ell, b)$ map, the $\ell$-$v$ strip at $b \approx 0$, and the velocity axis. The science notebook loads this pickle and proceeds with tangent-point fitting, gravitational and gas-mass derivations, kinematic-distance projection, and spiral-arm model overlay -- no further calibration code is touched downstream.

### Conclusion

The cool-method Y-factor calibration with the EBHIS-anchored pol-1 Fourier model produces a Galactic-plane $T_B$ cube whose integrated map agrees with EBHIS/LAB references to $\lesssim 10\%$ and whose $\ell$-$v$ strip shows all the kinematic features expected for a flat disk with a $\approx 220$ km/s rotation curve. The persisted data products are ready for direct consumption by the science notebook, with the optically-thin and pure-circular-rotation caveats explicitly flagged for downstream interpretation.


## Report figures (saved to `report/figures/`)


## Report figures (saved to `report/figures/`)


## Report figures (saved to `report/figures/`)


## Report figures (saved to `report/figures/`)


## Report figures (saved to `report/figures/`)


In [ ]:
# === Report figures (auto-saved to report/figures/) ===
from plotters import (savefig, plot_survey_mollweide_gridded,
                      plot_lv_strip)

# Beam-weighted W(l, b) gridded Mollweide.
fig, _ = plot_survey_mollweide_gridded(
    gl_arr[valid], gb_arr[valid], W_TB[valid],
    center_l=MOLL_CENTER_L,
    title='',
    cbar_label=r'$W = \int T_B\,dv$ [K km/s]',
    cmap='inferno', hpbw_deg=HPBW_DEG, pixel_deg=PIXEL_DEG,
    cutoff_hpbw=CUTOFF_HPBW,
)
savefig(fig, 'fig_W_mollweide.pdf')

# l-v strip at |b|<=4 deg.
fig, ax = plot_lv_strip(
    strip['l_fine'], v_lsr_overlap, strip['lv_image'],
    title='', cbar_label=r'$T_B$ [K]',
)
for l_a, v_a, txt in [(40, 110, 'tangent envelope'),
                       (80, -15, 'Local Arm'),
                       (140, -55, 'Perseus Arm'),
                       (200, -85, 'outer Galaxy'),
                       (-5, -95, 'GC anomaly')]:
    ax.annotate(txt, xy=(l_a, v_a), color='white', fontsize=8,
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.18', fc='black',
                          ec='white', alpha=0.5))
savefig(fig, 'fig_lv_strip.pdf')
